# Logistic Regression — Skenario A-Manual
**Data:** Manual 18.534 → Train:14.827 | Test:3.707  
**Referensi:** Parameter default sklearn — C=1.0, solver=lbfgs, max_iter=1000


## Cell 1 — Setup

In [10]:
import pandas as pd, numpy as np, os, pickle, time, warnings
warnings.filterwarnings('ignore')
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
    recall_score, classification_report, confusion_matrix)
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2

BASE_DIR   = r'C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method'
SPLIT_DIR  = os.path.join(BASE_DIR, 'splits')
MODEL_DIR  = os.path.join(BASE_DIR, 'models')
RESULT_DIR = os.path.join(BASE_DIR, 'results')
for d in [MODEL_DIR, RESULT_DIR]: os.makedirs(d, exist_ok=True)

SEED=42; LABEL_MAP={'keluhan':0,'saran':1,'pujian':2}
INV_MAP={v:k for k,v in LABEL_MAP.items()}; CLASS_NAMES=['keluhan','saran','pujian']

def evaluate(y_true, y_pred, exp_code, prefix):
    acc=accuracy_score(y_true,y_pred)
    mac=f1_score(y_true,y_pred,average='macro',zero_division=0)
    f1s=f1_score(y_true,y_pred,average=None,labels=[0,1,2],zero_division=0)
    prec=precision_score(y_true,y_pred,average=None,labels=[0,1,2],zero_division=0)
    rec=recall_score(y_true,y_pred,average=None,labels=[0,1,2],zero_division=0)
    rep=classification_report(y_true,y_pred,target_names=CLASS_NAMES,digits=4,zero_division=0)
    print(f'\n{"="*60}'); print(f'HASIL — {exp_code}'); print(f'{"="*60}')
    print(f'  Accuracy   : {acc*100:.2f}%')
    print(f'  Macro F1   : {mac:.4f}')
    print(f'  F1 Keluhan : {f1s[0]:.4f}  Prec:{prec[0]:.4f}  Rec:{rec[0]:.4f}')
    print(f'  F1 Saran   : {f1s[1]:.4f}  Prec:{prec[1]:.4f}  Rec:{rec[1]:.4f}')
    print(f'  F1 Pujian  : {f1s[2]:.4f}  Prec:{prec[2]:.4f}  Rec:{rec[2]:.4f}')
    print(f'\n{rep}')
    cm=confusion_matrix(y_true,y_pred,labels=[0,1,2])
    fig,ax=plt.subplots(figsize=(6,5)); vmax=cm.max()
    im=ax.imshow(cm,cmap='Blues',vmin=0,vmax=vmax)
    for i in range(3):
        for j in range(3):
            c='white' if cm[i,j]>vmax*0.55 else '#1A1A1A'
            ax.text(j,i,f'{cm[i,j]:,}',ha='center',va='center',fontsize=12,fontweight='bold',color=c)
    ax.set_xticks([0,1,2]); ax.set_xticklabels(CLASS_NAMES)
    ax.set_yticks([0,1,2]); ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel('Predicted',fontweight='bold'); ax.set_ylabel('Actual',fontweight='bold')
    ax.set_title(f'{exp_code}\nAcc={acc*100:.2f}% | MacroF1={mac:.4f}',fontweight='bold',pad=10)
    plt.colorbar(im,ax=ax,shrink=0.85); fig.tight_layout()
    path=os.path.join(RESULT_DIR,f'CM_{prefix}.png')
    fig.savefig(path); plt.close(); print(f'  CM → {path}')
    return {'exp':exp_code,'accuracy':acc,'macro_f1':mac,
            'f1_keluhan':f1s[0],'f1_saran':f1s[1],'f1_pujian':f1s[2]}

def tfidf_chi2(X_train, X_test, y_train):
    tfidf=TfidfVectorizer(ngram_range=(1,1),max_features=50000,
                          sublinear_tf=True,min_df=2,strip_accents='unicode')
    Xt=tfidf.fit_transform(X_train); Xte=tfidf.transform(X_test)
    k=min(1500,Xt.shape[1]); sel=SelectKBest(chi2,k=k)
    Xts=sel.fit_transform(Xt,y_train); Xtes=sel.transform(Xte)
    print(f'TF-IDF: {Xt.shape[1]:,} → Chi2: {k:,} fitur')
    return Xts, Xtes, tfidf, sel

print('Setup selesai!')


Setup selesai!


## Cell 2 — Load Data

In [11]:
df_train = pd.read_csv(os.path.join(SPLIT_DIR,'A_manual_xgb_train.csv'))
df_test  = pd.read_csv(os.path.join(SPLIT_DIR,'A_manual_xgb_test.csv'))
df_train['label_enc']=df_train['label_pks'].map(LABEL_MAP)
df_test['label_enc'] =df_test['label_pks'].map(LABEL_MAP)
X_train=df_train['text'].fillna('').values; y_train=df_train['label_enc'].values
X_test =df_test['text'].fillna('').values;  y_test =df_test['label_enc'].values
print(f'Skenario: A-Manual | Train:{len(X_train):,} | Test:{len(X_test):,}')
for l,c in zip(*np.unique(y_train,return_counts=True)):
    print(f'  {INV_MAP[l]:10s}: {c:,} ({c/len(y_train)*100:.1f}%)')

Skenario: A-Manual | Train:11,992 | Test:2,999
  keluhan   : 7,867 (65.6%)
  saran     : 2,136 (17.8%)
  pujian    : 1,989 (16.6%)


## Cell 3 — TF-IDF + Chi-Square

In [12]:
Xtr_sel,Xte_sel,tfidf,sel=tfidf_chi2(X_train,X_test,y_train)

TF-IDF: 2,648 → Chi2: 1,500 fitur


## Cell 4 — Training LR

In [13]:
from sklearn.linear_model import LogisticRegression
t0=time.time()
model=LogisticRegression(multi_class='multinomial',solver='lbfgs',
    max_iter=1000,class_weight='balanced',C=1.0,random_state=SEED)
model.fit(Xtr_sel,y_train)
print(f'Training: {time.time()-t0:.1f} detik')
y_pred=model.predict(Xte_sel)
result=evaluate(y_test,y_pred,'LR-A-Manual','LR_A_Manual')
pickle.dump(model,open(os.path.join(MODEL_DIR,'LR_A_Manual.pkl'),'wb'))
pickle.dump({'tfidf':tfidf,'selector':sel},open(os.path.join(MODEL_DIR,'vec_LR_A_Manual.pkl'),'wb'))
print('Model tersimpan!')

Training: 0.1 detik

HASIL — LR-A-Manual
  Accuracy   : 79.13%
  Macro F1   : 0.7582
  F1 Keluhan : 0.8499  Prec:0.9100  Rec:0.7972
  F1 Saran   : 0.5893  Prec:0.5026  Rec:0.7121
  F1 Pujian  : 0.8355  Prec:0.8185  Rec:0.8531

              precision    recall  f1-score   support

     keluhan     0.9100    0.7972    0.8499      1967
       saran     0.5026    0.7121    0.5893       535
      pujian     0.8185    0.8531    0.8355       497

    accuracy                         0.7913      2999
   macro avg     0.7437    0.7875    0.7582      2999
weighted avg     0.8222    0.7913    0.8010      2999

  CM → C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method\results\CM_LR_A_Manual.png
Model tersimpan!


## Cell 5 — Top 10 Fitur

In [14]:
feat=tfidf.get_feature_names_out()[sel.get_support()]
scores=sel.scores_[sel.get_support()]; top10=np.argsort(scores)[::-1][:10]
print(f'Top 10 Fitur — LR A-Manual:')
for i,idx in enumerate(top10):
    print(f'  {i+1:2d}. {feat[idx]:25s}: {scores[idx]:.2f}')

Top 10 Fitur — LR A-Manual:
   1. tepuk                    : 747.18
   2. apresiasi                : 744.85
   3. puji                     : 744.07
   4. tangan                   : 656.08
   5. gila                     : 575.11
   6. cinta                    : 415.70
   7. hati                     : 355.21
   8. keren                    : 297.88
   9. mata                     : 280.27
  10. cek                      : 277.41
